In [ ]:
import asyncio
import json
import websockets

HOST = "0.0.0.0"
PORT = 8765


def mux(d0, d1, d2, d3, selector):
    inputs = [d0, d1, d2, d3]
    return inputs[selector]


def demux(d, selector):
    outputs = [0, 0, 0, 0]
    outputs[selector] = d
    return outputs


async def handle_client(websocket):
    print("Client connected")

    try:
        async for message in websocket:
            data = json.loads(message)

            if data["type"] == "mux":
                y = mux(
                    data["d0"],
                    data["d1"],
                    data["d2"],
                    data["d3"],
                    data["selector"],
                )

                response = {
                    "type": "mux_state",
                    "d0": data["d0"],
                    "d1": data["d1"],
                    "d2": data["d2"],
                    "d3": data["d3"],
                    "selector": data["selector"],
                    "y": y,
                }

            elif data["type"] == "demux":
                outputs = demux(
                    data["d"],
                    data["selector"],
                )

                response = {
                    "type": "demux_state",
                    "d": data["d"],
                    "selector": data["selector"],
                    "outputs": outputs,
                }

            else:
                response = {
                    "type": "error",
                    "message": "Unknown command"
                }

            await websocket.send(json.dumps(response))

    except websockets.ConnectionClosed:
        print("Client disconnected")


server = await websockets.serve(
    handle_client,
    HOST,
    PORT,
)

print(f"WebSocket server running on ws://{HOST}:{PORT}")

In [ ]:
import asyncio
import json
import websockets


async def test_mux():
    async with websockets.connect("ws://127.0.0.1:8765") as websocket:
        request = {
            "type": "mux",
            "d0": 0,
            "d1": 1,
            "d2": 0,
            "d3": 1,
            "selector": 1
        }

        await websocket.send(json.dumps(request))

        response = await websocket.recv()
        response = json.loads(response)

        print(response)

        assert response["type"] == "mux_state"
        assert response["selector"] == 1
        assert response["y"] == 1


await test_mux()

In [ ]:
async def test_all_mux_routes():
    inputs = [0, 1, 0, 1]

    async with websockets.connect("ws://127.0.0.1:8765") as websocket:
        for selector in range(4):
            request = {
                "type": "mux",
                "d0": inputs[0],
                "d1": inputs[1],
                "d2": inputs[2],
                "d3": inputs[3],
                "selector": selector,
            }

            await websocket.send(json.dumps(request))

            response = json.loads(await websocket.recv())

            expected = inputs[selector]

            print(
                f"selector={selector} "
                f"expected={expected} "
                f"received={response['y']}"
            )

            assert response["y"] == expected

await test_all_mux_routes()

In [ ]:
async def test_all_demux_routes():
    async with websockets.connect("ws://127.0.0.1:8765") as websocket:
        for selector in range(4):
            request = {
                "type": "demux",
                "d": 1,
                "selector": selector,
            }

            await websocket.send(json.dumps(request))

            response = json.loads(await websocket.recv())

            expected = [0, 0, 0, 0]
            expected[selector] = 1

            print(
                f"selector={selector} "
                f"expected={expected} "
                f"received={response['outputs']}"
            )

            assert response["outputs"] == expected

await test_all_demux_routes()